In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%%capture
!pip install accelerate peft pyvi pyarrow

In [ ]:
from pathlib import Path
import sys
import json
import time

import numpy as np
import pandas as pd
import torch

MODEL_NAME = "intfloat/multilingual-e5-large"
RUN_NAME = "multilingual-e5-large"
BASELINE_RUN_NAME = "multilingual-e5-large__chunked"

MODEL_KWARGS = {}
TOKENIZER_KWARGS = {}

QUERY_PREFIX = "query: "
PASSAGE_PREFIX = "passage: "
USE_WORD_SEGMENTATION = False
QUERY_PROMPT_NAME = None


MAX_PASSAGE_LENGTH = 512
MAX_QUERY_LENGTH = 64

USE_CHUNKING = True
CHUNK_SIZE = 512
CHUNK_OVERLAP = 64

BATCH_SIZE_PASSAGE = 32
BATCH_SIZE_QUERY = 128

LOSS_TYPE = 'marginmse'

TEACHER_SCORE_COLUMN = 'cross_encoder_score'

if LOSS_TYPE == 'mnrl':
  EFFECTIVE_BATCH_SIZE = 128
  MINI_BATCH_SIZE = 16
  NUMBER_OF_EPOCHS = 3
else:
  EFFECTIVE_BATCH_SIZE = 16
  MINI_BATCH_SIZE = None
  NUMBER_OF_EPOCHS = 10

LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1

NEGATIVES_USED = 4

USE_LORA = False
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

RANDOM_SEED = 42

BASE_DIR = Path("/content/drive/MyDrive/Legal_Sytem")
DATA_DIR = BASE_DIR / "01_Dataset/02_processed_data"
RETRIEVER_DIR = BASE_DIR / "02_Retriever"

TRAIN_DATA_DIR = RETRIEVER_DIR / "03_Finetuning/train_data"
BASELINE_OUTPUT_DIR = RETRIEVER_DIR / "02_Baseline/BiEncoder/output"

OUTPUT_DIR = RETRIEVER_DIR / "03_Finetuning/output"
CHECKPOINT_DIR = RETRIEVER_DIR / "03_Finetuning/checkpoints" / RUN_NAME
MODEL_SAVE_DIR = RETRIEVER_DIR / "03_Finetuning/models" / RUN_NAME

for directory in [OUTPUT_DIR, CHECKPOINT_DIR, MODEL_SAVE_DIR]:
  directory.mkdir(parents=True, exist_ok=True)

sys.path.append(str(RETRIEVER_DIR))

import retriever_common as rc

_LOSS_TAG = 'mnrl' if LOSS_TYPE == 'mnrl' else f"marginmse-{TEACHER_SCORE_COLUMN.replace('_score', '')}"

FINETUNED_RUN_NAME = f"{RUN_NAME}__{_LOSS_TAG}"

torch.manual_seed(RANDOM_SEED)

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
  print("GPU:", torch.cuda.get_device_name(0))
  print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


## **01. Read data**

In [5]:
corpus = pd.read_parquet(DATA_DIR / 'corpus' / 'corpus.parquet')
query = pd.read_parquet(DATA_DIR / 'corpus' / 'queries.parquet')
qrels = pd.read_parquet(DATA_DIR / 'corpus' / 'qrels.parquet')

training_examples = pd.read_parquet(TRAIN_DATA_DIR / 'train_examples.parquet')

with open(TRAIN_DATA_DIR / 'mining_config.json', encoding='utf-8') as file:
  mining_config = json.load(file)

print(f"Corpus         : {len(corpus):,} passage")
print(f"Mẫu huấn luyện : {len(training_examples):,}")
print(f"LOSS_TYPE      : {LOSS_TYPE}")

if LOSS_TYPE == 'mnrl':
  print(f"Trần batch an toàn: {mining_config['max_safe_batch_size']}")

  assert EFFECTIVE_BATCH_SIZE <= mining_config['max_safe_batch_size'], (
      f"EFFECTIVE_BATCH_SIZE ({EFFECTIVE_BATCH_SIZE}) vượt số passage phân biệt "
      f"({mining_config['max_safe_batch_size']}) — bộ lấy mẫu NO_DUPLICATES sẽ không xếp nổi batch."
  )

  teacher_scores = None
else:
  teacher_scores = pd.read_parquet(TRAIN_DATA_DIR / 'teacher_scores.parquet')

  assert TEACHER_SCORE_COLUMN in teacher_scores.columns, (
      f"Không có cột '{TEACHER_SCORE_COLUMN}'. Các cột có sẵn: {teacher_scores.columns.tolist()}"
  )

  print(f"Teacher scores : {len(teacher_scores):,} cặp, dùng cột '{TEACHER_SCORE_COLUMN}'")

Corpus         : 294 passage
Mẫu huấn luyện : 7,718
LOSS_TYPE      : marginmse
Teacher scores : 69,294 cặp, dùng cột 'cross_encoder_score'


In [6]:
from pyvi import ViTokenizer

def prepare_passage(text: str) -> str:
  text = str(text)

  if USE_WORD_SEGMENTATION:
    text = ViTokenizer.tokenize(text)

  return PASSAGE_PREFIX + text


def prepare_query(text: str) -> str:
  text = str(text)

  if USE_WORD_SEGMENTATION:
    text = ViTokenizer.tokenize(text)

  return QUERY_PREFIX + text

## **02. Build train data**

In [ ]:
from datasets import Dataset

doc_id_to_text = dict(zip(corpus['doc_id'], corpus['embedding_text']))

prepared_passage_cache = {
    doc_id: prepare_passage(text) for doc_id, text in doc_id_to_text.items()
}

In [8]:
def build_mnrl_dataset() -> Dataset:
  columns = {'anchor': [], 'positive': []}

  for negative_index in range(NEGATIVES_USED):
    columns[f'negative_{negative_index + 1}'] = []

  for row in training_examples.itertuples(index=False):
    negative_doc_ids = list(row.negative_doc_ids)[:NEGATIVES_USED]

    if len(negative_doc_ids) < NEGATIVES_USED:
      continue

    columns['anchor'].append(prepare_query(row.query))
    columns['positive'].append(prepared_passage_cache[row.positive_doc_id])

    for negative_index, doc_id in enumerate(negative_doc_ids):
      columns[f'negative_{negative_index + 1}'].append(prepared_passage_cache[doc_id])

  return Dataset.from_dict(columns)


def build_marginmse_dataset() -> Dataset:
  score_lookup = dict(
      zip(zip(teacher_scores['query_id'], teacher_scores['doc_id']),
          teacher_scores[TEACHER_SCORE_COLUMN])
  )

  columns = {'query': [], 'positive': [], 'negative': [], 'label': []}

  skipped = 0

  for row in training_examples.itertuples(index=False):
    positive_score = score_lookup.get((row.query_id, row.positive_doc_id))

    if positive_score is None:
      skipped += 1
      continue

    for doc_id in list(row.negative_doc_ids)[:NEGATIVES_USED]:
      negative_score = score_lookup.get((row.query_id, doc_id))

      if negative_score is None:
        skipped += 1
        continue

      columns['query'].append(prepare_query(row.query))
      columns['positive'].append(prepared_passage_cache[row.positive_doc_id])
      columns['negative'].append(prepared_passage_cache[doc_id])
      columns['label'].append(float(positive_score) - float(negative_score))

  if skipped:
    print(f"Bỏ qua {skipped:,} cặp thiếu điểm teacher.")

  return Dataset.from_dict(columns)

In [9]:
train_dataset = build_mnrl_dataset() if LOSS_TYPE == 'mnrl' else build_marginmse_dataset()

print(train_dataset)

if LOSS_TYPE == 'marginmse':
  labels = np.asarray(train_dataset['label'])

  print(f"\nMargin: mean={labels.mean():.4f}  p05={np.percentile(labels, 5):.4f}  "
        f"p95={np.percentile(labels, 95):.4f}  max_abs={np.abs(labels).max():.4f}")
  print(f"Tỷ lệ margin âm (teacher chấm negative cao hơn positive): {(labels < 0).mean():.4f}")
  print("\nMargin âm KHÔNG phải lỗi — đó là cơ chế chống false negative của MarginMSE.")

first_example = train_dataset[0]

print("\nMẫu đầu tiên:")

for key, value in first_example.items():
  print(f"  {key:11s}:", (value[:130] if isinstance(value, str) else value))

Dataset({
    features: ['query', 'positive', 'negative', 'label'],
    num_rows: 30872
})

Margin: mean=0.8057  p05=0.1706  p95=0.9962  max_abs=1.0000
Tỷ lệ margin âm (teacher chấm negative cao hơn positive): 0.0049

Margin âm KHÔNG phải lỗi — đó là cơ chế chống false negative của MarginMSE.

Mẫu đầu tiên:
  query      : query: Sinh viên dự bị không trở thành sinh viên chính thức bao nhiêu học kỳ sẽ bị loại khỏi CTTN?
  positive   : passage: Văn bản: QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH TÀI NĂNG
Điều  9.	Tuyển bổ sung và loại ra khỏi chương trình, xét chính thức và dự
  negative   : passage: Văn bản: QUY CHẾ ĐÀO TẠO THEO HỌC CHẾ TÍN CHỈ CHO HỆ ĐẠI HỌC CHÍNH QUY CỦA TRƯỜNG ĐẠI HỌC CÔNG NGHỆ THÔNG TIN
Điều  18. H
  label      : 0.9819564819335938


## **03. Import model**

In [10]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    MODEL_NAME,
    model_kwargs=MODEL_KWARGS,
    tokenizer_kwargs=TOKENIZER_KWARGS,
    trust_remote_code=True,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

model.max_seq_length = MAX_PASSAGE_LENGTH

print("Model:", MODEL_NAME)
print("Embedding dimension:", model.get_sentence_embedding_dimension())
print("padding_side:", model.tokenizer.padding_side)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

model.safetensors: downloading bytes:           |  0.00B            

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Model: intfloat/multilingual-e5-large
Embedding dimension: 1024
padding_side: right


/tmp/ipykernel_10309/3470145641.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


In [11]:
if USE_LORA:
  from peft import LoraConfig, TaskType

  lora_config = LoraConfig(
      task_type=TaskType.FEATURE_EXTRACTION,
      r=LORA_R,
      lora_alpha=LORA_ALPHA,
      lora_dropout=LORA_DROPOUT,
      target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
  )

  model.add_adapter(lora_config)

  trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
  total_parameters = sum(p.numel() for p in model.parameters())

  print(f"Tham số huấn luyện: {trainable_parameters:,} / {total_parameters:,} "
        f"({trainable_parameters / total_parameters * 100:.2f}%)")
else:
  print("Full fine-tune — huấn luyện toàn bộ tham số.")

Full fine-tune — huấn luyện toàn bộ tham số.


## **04. Config**

In [ ]:
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import CachedMultipleNegativesRankingLoss, MarginMSELoss
from sentence_transformers.training_args import BatchSamplers

if LOSS_TYPE == 'mnrl':
  loss = CachedMultipleNegativesRankingLoss(model, mini_batch_size=MINI_BATCH_SIZE)
else:
  # MarginMSELoss mặc định dùng dot product (KHÔNG chuẩn hoá vector). Đây là lựa chọn có lý:
  # margin của teacher có thể lên tới hàng chục, còn cosine similarity bị chặn trong [-1, 1]
  # nên không thể khớp được. Hệ quả: phải đánh giá bằng dot product, xem NORMALIZE_EMBEDDINGS.
  loss = MarginMSELoss(model)

# Với họ Qwen3-Embedding, instruction prompt của query phải được áp cả lúc huấn luyện.
training_prompts = None

if QUERY_PROMPT_NAME is not None and getattr(model, 'prompts', None):
  query_prompt_text = model.prompts.get(QUERY_PROMPT_NAME)

  if query_prompt_text:
    query_column = 'anchor' if LOSS_TYPE == 'mnrl' else 'query'
    training_prompts = {query_column: query_prompt_text}
    print(f"Prompt áp cho cột {query_column}:", repr(query_prompt_text))


batch_sampler = BatchSamplers.NO_DUPLICATES if LOSS_TYPE == 'mnrl' else BatchSamplers.BATCH_SAMPLER

training_arguments = SentenceTransformerTrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=NUMBER_OF_EPOCHS,
    per_device_train_batch_size=EFFECTIVE_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    bf16=torch.cuda.is_available(),
    batch_sampler=batch_sampler,
    prompts=training_prompts,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='eval_val_mrr@10',
    greater_is_better=True,
    logging_steps=10,
    seed=RANDOM_SEED,
    report_to='none',
)

/tmp/ipykernel_10309/3757839752.py:2: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import CachedMultipleNegativesRankingLoss, MarginMSELoss
/tmp/ipykernel_10309/3757839752.py:3: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import BatchSamplers


### **5.1. Evaluation**

In [ ]:
from sentence_transformers.evaluation import SentenceEvaluator


NORMALIZE_EMBEDDINGS = (LOSS_TYPE == 'mnrl')

print("normalize_embeddings khi đánh giá:", NORMALIZE_EMBEDDINGS,
      "(cosine)" if NORMALIZE_EMBEDDINGS else "(dot product)")

EVALUATION_KWARGS = dict(
    normalize_embeddings=NORMALIZE_EMBEDDINGS,
    use_chunking=USE_CHUNKING,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    max_passage_length=MAX_PASSAGE_LENGTH,
    max_query_length=MAX_QUERY_LENGTH,
    batch_size_passage=BATCH_SIZE_PASSAGE,
    batch_size_query=BATCH_SIZE_QUERY,
    query_prompt_name=QUERY_PROMPT_NAME,
)

REPORTED_METRICS = ['recall@1', 'recall@3', 'recall@5', 'recall@10', 'recall@15', 'recall@20', 'mrr@1', 'mrr@10', 'ndcg@1', 'ndcg@10']


class RetrievalEvaluator(SentenceEvaluator):
  def __init__(self, split_name: str = 'val'):
    super().__init__()
    self.split_name = split_name
    self.eval_query = query[query['split'] == split_name]
    self.eval_qrels = qrels[qrels['split'] == split_name]
    self.primary_metric = f'{split_name}_mrr@10'

  def __call__(self, model, output_path=None, epoch=-1, steps=-1) -> dict:
    _, per_query_dataframe, _, _ = rc.evaluate_bi_encoder(
        model, corpus, self.eval_query, self.eval_qrels,
        prepare_query, prepare_passage, split_name=self.split_name, **EVALUATION_KWARGS,
    )

    summary = rc.summarize(per_query_dataframe)

    metrics = {f'{self.split_name}_{name}': float(summary[name]) for name in REPORTED_METRICS}

    print(f"[epoch {epoch}] " + "  ".join(f"{name}={value:.4f}" for name, value in metrics.items()))

    return metrics


validation_evaluator = RetrievalEvaluator('val')

normalize_embeddings khi đánh giá: False (dot product)


/tmp/ipykernel_10309/614173896.py:1: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import SentenceEvaluator


## **06. Training**

In [14]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=training_arguments,
    train_dataset=train_dataset,
    loss=loss,
    evaluator=validation_evaluator,
)

start_time = time.perf_counter()

trainer.train()

training_seconds = time.perf_counter() - start_time

print(f"\nHuấn luyện xong trong {training_seconds / 60:.1f} phút")

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Epoch,Training Loss,Validation Loss,Val Recall@1,Val Recall@3,Val Recall@5,Val Recall@10,Val Recall@15,Val Recall@20,Val Mrr@1,Val Mrr@10,Val Ndcg@1,Val Ndcg@10
1,0.103860,No log,0.617363,0.826367,0.873526,0.922830,0.944266,0.959271,0.617363,0.727895,0.617363,0.775779
2,0.083214,No log,0.635584,0.840300,0.891747,0.933548,0.947481,0.954984,0.635584,0.744938,0.635584,0.791435
3,0.064642,No log,0.661308,0.844587,0.893891,0.938907,0.952840,0.958199,0.661308,0.762386,0.661308,0.805747
4,0.028683,No log,0.682744,0.857449,0.907824,0.934620,0.951768,0.962487,0.682744,0.778291,0.682744,0.817015
5,0.024475,No log,0.703108,0.870311,0.916399,0.947481,0.957128,0.960343,0.703108,0.794135,0.703108,0.832024
6,0.022388,No log,0.724544,0.889603,0.920686,0.950697,0.959271,0.961415,0.724544,0.811401,0.724544,0.845905
7,0.018691,No log,0.730975,0.886388,0.924973,0.952840,0.958199,0.963558,0.730975,0.815421,0.730975,0.849435
8,0.013551,No log,0.730975,0.892819,0.926045,0.952840,0.960343,0.963558,0.730975,0.816412,0.730975,0.850284
9,0.011184,No log,0.734191,0.899250,0.928189,0.952840,0.961415,0.963558,0.734191,0.819918,0.734191,0.852992
10,0.011382,No log,0.742765,0.894962,0.929260,0.949625,0.958199,0.962487,0.742765,0.824356,0.742765,0.855570


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (963 > 512). Running this sequence through the model will result in indexing errors


[epoch 1.0] val_recall@1=0.6174  val_recall@3=0.8264  val_recall@5=0.8735  val_recall@10=0.9228  val_recall@15=0.9443  val_recall@20=0.9593  val_mrr@1=0.6174  val_mrr@10=0.7279  val_ndcg@1=0.6174  val_ndcg@10=0.7758


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[epoch 2.0] val_recall@1=0.6356  val_recall@3=0.8403  val_recall@5=0.8917  val_recall@10=0.9335  val_recall@15=0.9475  val_recall@20=0.9550  val_mrr@1=0.6356  val_mrr@10=0.7449  val_ndcg@1=0.6356  val_ndcg@10=0.7914


... (15 outputs left out)

[epoch 10.0] val_recall@1=0.7428  val_recall@3=0.8950  val_recall@5=0.9293  val_recall@10=0.9496  val_recall@15=0.9582  val_recall@20=0.9625  val_mrr@1=0.7428  val_mrr@10=0.8244  val_ndcg@1=0.7428  val_ndcg@10=0.8556


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


Huấn luyện xong trong 89.6 phút


In [15]:
model.save_pretrained(str(MODEL_SAVE_DIR))

print("Đã lưu model:", MODEL_SAVE_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã lưu model: /content/drive/MyDrive/Legal_Sytem/02_Retriever/03_Finetuning/models/multilingual-e5-large


## **07. Evaluation**

In [16]:
ranking_by_split = {}
per_query_by_split = {}

passage_embeddings = None
chunk_table = None

for split_name in ['val', 'test']:
  split_ranking, split_per_query, passage_embeddings, chunk_table = rc.evaluate_bi_encoder(
      model, corpus,
      query[query['split'] == split_name],
      qrels[qrels['split'] == split_name],
      prepare_query, prepare_passage,
      split_name=split_name,
      passage_embeddings=passage_embeddings,
      chunk_table=chunk_table,
      **EVALUATION_KWARGS,
  )

  rc.assert_metrics_are_sane(split_per_query)

  ranking_by_split[split_name] = split_ranking
  per_query_by_split[split_name] = split_per_query

  summary = rc.summarize(split_per_query)

  print(f"[{split_name}] " + "  ".join(f"{name}={summary[name]:.4f}" for name in REPORTED_METRICS))

[val] recall@1=0.7428  recall@3=0.8950  recall@5=0.9293  recall@10=0.9496  recall@15=0.9582  recall@20=0.9625  mrr@1=0.7428  mrr@10=0.8244  ndcg@1=0.7428  ndcg@10=0.8556
[test] recall@1=0.7338  recall@3=0.8935  recall@5=0.9207  recall@10=0.9541  recall@15=0.9624  recall@20=0.9687  mrr@1=0.7338  mrr@10=0.8165  ndcg@1=0.7338  ndcg@10=0.8505


### **7.1. So vs zero-shot**

In [17]:
baseline_metrics_path = BASELINE_OUTPUT_DIR / f'{BASELINE_RUN_NAME}__val_metrics_per_query.parquet'

if baseline_metrics_path.exists():
  baseline_per_query = pd.read_parquet(baseline_metrics_path)

  comparison_records = []

  for metric in ['recall@1', 'recall@3', 'recall@5', 'recall@10', 'mrr@10', 'ndcg@10']:
    result = rc.paired_bootstrap(per_query_by_split['val'], baseline_per_query, metric=metric)

    comparison_records.append({
        'metric': metric,
        'finetuned': round(result['mean_a'], 4),
        'zero_shot': round(result['mean_b'], 4),
        'difference': round(result['difference'], 4),
        'ci_95': f"[{result['ci_lower_95']:+.4f}, {result['ci_upper_95']:+.4f}]",
        'p_value': round(result['p_value'], 4),
        'significant': result['significant_at_0.05'],
    })

  display(pd.DataFrame(comparison_records))
else:
  print("Không tìm thấy per-query metrics của bản zero-shot:", baseline_metrics_path)
  print("Chạy notebook baseline tương ứng trước để có bảng so sánh này.")

Không tìm thấy per-query metrics của bản zero-shot: /content/drive/MyDrive/Legal_Sytem/02_Retriever/02_Baseline/BiEncoder/output/multilingual-e5-large__chunked__val_metrics_per_query.parquet
Chạy notebook baseline tương ứng trước để có bảng so sánh này.


## **08. Latency**

In [18]:
doc_id_array = corpus['doc_id'].to_numpy()

model.max_seq_length = MAX_QUERY_LENGTH

latency_query_list = query[query['split'] == 'val']['query'].tolist()[:200]


def encode_single_query(query_text: str) -> np.ndarray:
  encode_kwargs = dict(normalize_embeddings=NORMALIZE_EMBEDDINGS,
                       convert_to_numpy=True, show_progress_bar=False)

  if QUERY_PROMPT_NAME is not None:
    encode_kwargs['prompt_name'] = QUERY_PROMPT_NAME

  return model.encode([prepare_query(query_text)], **encode_kwargs).astype(np.float32)


def search_single_query(query_vector: np.ndarray):
  score_matrix = query_vector @ passage_embeddings.T

  if USE_CHUNKING:
    score_matrix = rc.max_pool_chunk_scores(score_matrix, chunk_table, doc_id_array)

  return np.argsort(-score_matrix[0], kind='stable')[:rc.RANKING_DEPTH]


latency_report = rc.measure_latency(encode_single_query, search_single_query, latency_query_list)

latency_report['peak_vram_gb'] = (
    round(torch.cuda.max_memory_allocated() / 1e9, 2) if torch.cuda.is_available() else None
)

print(json.dumps(latency_report, ensure_ascii=False, indent=2))

{
  "number_of_queries": 200,
  "encode_ms_per_query": 26.956,
  "search_ms_per_query": 0.857,
  "latency_ms_per_query": 27.813,
  "queries_per_second": 36.0,
  "peak_vram_gb": 27.68
}


## **09. Save**

In [19]:
run_config = {
    'run_name': FINETUNED_RUN_NAME,
    'model_name': MODEL_NAME,
    'baseline_run_name': BASELINE_RUN_NAME,
    'finetuned': True,
    'training_method': 'lora' if USE_LORA else 'full',
    'lora_r': LORA_R if USE_LORA else None,
    'lora_alpha': LORA_ALPHA if USE_LORA else None,
    'loss_type': LOSS_TYPE,
    'loss': 'CachedMultipleNegativesRankingLoss' if LOSS_TYPE == 'mnrl' else 'MarginMSELoss',
    'teacher_score_column': TEACHER_SCORE_COLUMN if LOSS_TYPE == 'marginmse' else None,
    'batch_sampler': 'NO_DUPLICATES' if LOSS_TYPE == 'mnrl' else 'default',
    'normalize_embeddings': NORMALIZE_EMBEDDINGS,
    'effective_batch_size': EFFECTIVE_BATCH_SIZE,
    'mini_batch_size': MINI_BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'number_of_epochs': NUMBER_OF_EPOCHS,
    'negatives_used': NEGATIVES_USED,
    'training_examples': len(train_dataset),
    'training_minutes': round(training_seconds / 60, 2),
    'use_chunking': USE_CHUNKING,
    'chunk_size': CHUNK_SIZE if USE_CHUNKING else None,
    'max_passage_length': MAX_PASSAGE_LENGTH,
    'query_prefix': QUERY_PREFIX,
    'passage_prefix': PASSAGE_PREFIX,
    'use_word_segmentation': USE_WORD_SEGMENTATION,
    'query_prompt_name': QUERY_PROMPT_NAME,
    'corpus_size': len(corpus),
    'mining_config': mining_config,
    **latency_report,
}

summary_dataframe = rc.save_run(
    OUTPUT_DIR, FINETUNED_RUN_NAME, ranking_by_split, per_query_by_split, run_config
)

display(summary_dataframe[['run_name', 'split'] + REPORTED_METRICS].round(4))

,run_name,split,recall@1,recall@3,recall@5,recall@10,recall@15,recall@20,mrr@1,mrr@10,ndcg@1,ndcg@10
0,multilingual-e5-large__marginmse-cross_encoder,val,0.7428,0.8950,0.9293,0.9496,0.9582,0.9625,0.7428,0.8244,0.7428,0.8556
1,multilingual-e5-large__marginmse-cross_encoder,test,0.7338,0.8935,0.9207,0.9541,0.9624,0.9687,0.7338,0.8165,0.7338,0.8505


In [20]:
leaderboard = rc.collect_leaderboard(OUTPUT_DIR)

display(leaderboard[leaderboard['split'] == 'val'].round(4))

,run_name,split,number_of_queries,p@1,p@5,p@10,p@15,p@20,recall@1,recall@5,recall@10,mrr@1,mrr@10,ndcg@1,ndcg@10
5,multilingual-e5-large__marginmse-cross_encoder,val,933,0.7428,0.9293,0.9496,0.9582,0.9625,0.7428,0.9293,0.9496,0.7428,0.8244,0.7428,0.8556
6,Qwen3-Embedding-0.6B__marginmse-cross_encoder,val,933,NaN,NaN,NaN,NaN,NaN,0.6270,0.8832,0.9218,0.6270,0.7324,0.6270,0.7789
7,Qwen3-Embedding-4B__marginmse-cross_encoder,val,933,NaN,NaN,NaN,NaN,NaN,0.5734,0.8264,0.8864,0.5734,0.6829,0.5734,0.7324
8,bge-m3__marginmse-cross_encoder,val,933,NaN,NaN,NaN,NaN,NaN,0.6688,0.9057,0.9346,0.6688,0.7683,0.6688,0.8094
9,vietnamese-bi-encoder__marginmse-cross_encoder,val,933,NaN,NaN,NaN,NaN,NaN,0.6977,0.9550,0.9775,0.6977,0.8105,0.6977,0.8522
